# Transmon

The transmon is a Josephson junction shunted by a large capacitance [Koch et al., Phys. Rev. A **76**, 042319 (2007)](https://doi.org/10.1103/PhysRevA.76.042319).
Its single node is compact: the junction is the only path to ground, so the node charge is an integer number of Cooper pairs.

In [ ]:
import sympy as sym
import circuit_quantization as cq
from circuit_quantization import Branch, Circuit, Loops, quantize

## Circuit

Capacitances and inductances are given in farads and henries, so `ghz_to_farad` converts a charging energy $E_C$ in GHz.
The junction's capacitance `C` is the total shunt capacitance.

In [ ]:
E_C, E_J = sym.symbols("E_C E_J", positive=True)

transmon = Circuit([Branch("J", 1, 0, C=cq.ghz_to_farad(E_C), EJ=E_J)], ground=0)
cq.draw_circuit_diagram(transmon, grounds="below")

## Quantization

With no inductive loop the loop basis is empty, and `quantize` returns the Hamiltonian $H/h$ in GHz.

In [ ]:
result = quantize(transmon, Loops.from_circuit(transmon), simplify=True)
assert result.coords.types == ("c",)
result

The offset charge `n_g1` appears automatically because the coordinate is compact.

## Gate voltage

A gate capacitor $C_g$ to a voltage source $V_g$ adds the gate charge $C_g V_g / 2e$ to the offset charge and renormalizes the charging energy.
The loop through the source is capacitively broken, so `zero_capacitive=True` sets its flux to zero.

In [ ]:
C_g, V_g = sym.symbols("C_g V_g", positive=True)

gated = Circuit([
    Branch("J", 1, 0, C=cq.ghz_to_farad(E_C), EJ=E_J),
    Branch("Cg", 1, "g", C=C_g),
    Branch("Vg", "g", 0, V=V_g),
], ground=0)
cq.draw_circuit_diagram(gated, layout={1: (0, 0), "g": (1, 0)}, grounds="below")

In [ ]:
gated_result = quantize(gated, Loops.from_circuit(gated, zero_capacitive=True), simplify=True)
assert gated_result.coords.types == ("c",)
gated_result